# Lab 5: Singular Value Decomposition

<a target="_blank" href="https://colab.research.google.com/github/drchadvidden/courseMaterials/blob/main/UnsupervisedLearning/Labs/Lab%205/Lab_5.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab Instructions

Run each of the coding cells. For tutorial example cells, understand the commands and check that the outputs make sense. For exercise cells, write your own code where indicated to generate the correct output. Give text explanations where indicated.

### Submission:
Complete the following notebook in order. Once done, save the notebook, print the file as a .pdf, and upload the resulting file to the Canvas course assignment.

### Rubric:
15 total points, 5 points to running tutorial example cells and saving outputs, 10 points for completing exercises.

### Deadline:
Tuesday at midnight after the lab is assigned.

# Tutorial: SVD explained

Here we illustrate how to use python to calculate and explore singular value decomposition (SVD).

## Singular Value Decomposition (SVD) calculation

The Singular Value Decomposition (SVD) factors any matrix
$X \in \mathbb{R}^{m \times n}$ as

$$
X = U \Sigma V^T
$$

where:

- $U$ is an orthogonal $m \times m$ matrix (left singular vectors),
- $\Sigma$ is an $m \times n$ diagonal matrix containing the singular values  
  $\sigma_1 \ge \sigma_2 \ge \cdots \ge 0$,
- $V$ is an orthogonal $n \times n$ matrix (right singular vectors).

Geometrically, SVD says every linear transformation can be viewed as:

rotation → scaling → rotation.

The singular values measure how strongly the matrix stretches space along orthogonal directions.

In unsupervised learning, SVD is fundamental for:
- PCA  
- dimensionality reduction  
- low-rank approximation  

Setting `full_matrices=False` computes the economy (reduced) SVD, which keeps only the essential components.

In [ ]:
import numpy as np

# Define the matrix
X = np.array([
    [3, 2,  2],
    [2, 3, -2]
], dtype=float)

# Compute full SVD, set parameter to false to see econdomy SVD
U, S, Vt = np.linalg.svd(X, full_matrices=True)

# Optional: reconstruct A to verify
Sigma = np.zeros((U.shape[0], Vt.shape[0]))
np.fill_diagonal(Sigma, S)

print("Left singular vectors: U =")
print(U)
print("\nSingular values: Sigma =")
print(Sigma)
print("\nRight singular vectors: V^T =")
print(Vt)

print("\nCheck orthogonality: U^T U =")
print(U.T @ U)

print("\nCheck orthogonality: V^T V =")
print(Vt.T @ Vt)

X_reconstructed = U @ Sigma @ Vt

print("\nX =")
print(X)
print("\nSVD Reconstructed X =")
print(X_reconstructed)

## Visualizing the Geometry of SVD

This example shows the geometric meaning of the singular value decomposition (SVD) for a 2×3 matrix $X$:

1. We start with a **unit sphere** in 3D (the input space of $X$).  
2. Mapping the sphere through $X$ produces a **2D ellipse** in the output space.  
3. The **principal axes** of the ellipse are the images of the right singular vectors scaled by their singular values: $X v_1$ and $X v_2$.  
4. The **lengths** of these axes correspond to the singular values $\sigma_1$ and $\sigma_2$, showing how the matrix stretches space along these directions.  

The plots and quivers make the SVD concept visually intuitive: rotation → stretching → rotation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the matrix
X = np.array([
    [3, 2,  2],
    [2, 3, -2]
], dtype=float)

# Compute SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)
v1 = Vt.T[:, 0]
v2 = Vt.T[:, 1]

# ---- Create unit sphere ----
phi = np.linspace(0, np.pi, 50)
theta = np.linspace(0, 2*np.pi, 50)
phi, theta = np.meshgrid(phi, theta)

x = np.sin(phi) * np.cos(theta)
y = np.sin(phi) * np.sin(theta)
z = np.cos(phi)

# Flatten sphere points
sphere_points = np.vstack([x.ravel(), y.ravel(), z.ravel()])

# ---- Map sphere through X ----
mapped = X @ sphere_points

# ---- Ellipse axes ----
Xv1 = X @ v1
Xv2 = X @ v2

# ---- Plot 1: Unit Sphere ----
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(x, y, z, alpha=0.6)
ax.set_title("Unit Sphere in R^3")
ax.set_box_aspect([1,1,1])
plt.show()

# ---- Plot 2: Ellipse in R^2 ----
plt.figure()
plt.scatter(mapped[0], mapped[1], s=1, alpha=0.5)
plt.gca().set_aspect('equal')

# Plot axes
origin = np.zeros(2)
plt.quiver(*origin, Xv1[0], Xv1[1], color='r', scale=1, scale_units='xy', width=0.005, label=r'$X v_1$')
plt.quiver(*origin, Xv2[0], Xv2[1], color='b', scale=1, scale_units='xy', width=0.005, label=r'$X v_2$')

plt.title("Image of Unit Sphere under X with principal axes")
plt.legend()
plt.show()

# See X V product result
print("\nFirst principal direction: X v_1 =")
print(Xv1)
print("\nSecond principal direction: X v_2 =")
print(Xv2)
print("\nFirst singular value: sigma_1=")
print(S[0])
print("\nSecond singular value: sigma_2=")
print(S[1])

## Low-Rank Approximation Using SVD

One of the most powerful uses of SVD is to **approximate a matrix by a lower-rank matrix**.  
The idea:

1. Compute SVD: $X = U \Sigma V^T$.
2. Keep only the largest $k$ singular values and corresponding singular vectors.
3. Reconstruct:
   $$
   X_k = \sum_{i=1}^k \sigma_i u_i v_i^T
   $$

This gives the **best rank-$k$ approximation** in the least-squares sense.  

Below we illustrate this with our 2×3 matrix $X$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the matrix
X = np.array([
    [3, 2, 2],
    [2, 3, -2],
], dtype=float)

# Compute full SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)

# ---- Rank-1 approximation ----
k = 1
Sigma_k = np.diag(S[:k])
U_k = U[:, :k]
Vt_k = Vt[:k, :]

X1 = U_k @ Sigma_k @ Vt_k

# ---- Rank-2 approximation ----
k = 2
Sigma_k2 = np.diag(S[:k])
U_k2 = U[:, :k]
Vt_k2 = Vt[:k, :]

X2 = U_k2 @ Sigma_k2 @ Vt_k2

# ---- Print results ----
print("Original X:\n", X)
print("\nRank-1 approximation X1:\n", X1)
print("\nRank-2 approximation X2:\n", X2)

# ---- Optional: error ----
err1 = np.linalg.norm(X - X1, 'fro')
err2 = np.linalg.norm(X - X2, 'fro')
print("\nFrobenius norm error: rank-1 =", err1, ", rank-2 =", err2)

# ---- Visualize with ellipses ----
# Unit sphere in 3D
phi = np.linspace(0, np.pi, 50)
theta = np.linspace(0, 2*np.pi, 50)
phi, theta = np.meshgrid(phi, theta)
x = np.sin(phi) * np.cos(theta)
y = np.sin(phi) * np.sin(theta)
z = np.cos(phi)
sphere_points = np.vstack([x.ravel(), y.ravel(), z.ravel()])

# Map sphere through original X
mapped_X = X @ sphere_points
mapped_X1 = X1 @ sphere_points
mapped_X2 = X2 @ sphere_points

# Plot comparison
plt.figure(figsize=(10,5))
plt.subplot(1,3,1)
plt.scatter(mapped_X[0], mapped_X[1], s=1, alpha=0.5)
plt.gca().set_aspect('equal')
plt.title("Original X")

plt.subplot(1,3,2)
plt.scatter(mapped_X1[0], mapped_X1[1], s=1, alpha=0.5)
plt.gca().set_aspect('equal')
plt.title("Rank-1 Approx")

plt.subplot(1,3,3)
plt.scatter(mapped_X2[0], mapped_X2[1], s=1, alpha=0.5)
plt.gca().set_aspect('equal')
plt.title("Rank-2 Approx")

plt.show()

We repeat this low rank approximation for a larger matrix ($10 \times 10$) to illustrate the matrix approximation error. Feel free to pring the matrices $X_k$ to see if they resemble $X$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a random 10x10 matrix
np.random.seed(42)
X = np.random.randn(10, 10)

#plt.imshow(X, cmap='viridis', interpolation='nearest')

# Compute full SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)

# Compute Frobenius norm error for rank-k approximations
errors = []
ranks = range(1, 11)
for k in ranks:
    # Rank-k approximation
    Xk = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    # Frobenius norm of the error
    err = np.linalg.norm(X - Xk, 'fro')
    errors.append(err)

#plt.imshow(Xk, cmap='viridis', interpolation='nearest')

# Plot the error vs rank
plt.figure(figsize=(6,4))
plt.plot(ranks, errors, marker='o')
plt.xticks(ranks)
plt.xlabel('Rank k')
plt.ylabel('Frobenius Norm of Error ||X - X_k||_F')
plt.title('Low-Rank Approximation Error vs Rank')
plt.grid(True)
plt.show()

## Tutorial Image Compression Using SVD

In this tutorial, we demonstrate how the **singular value decomposition (SVD)** can be used to compress an image:

1. Treat the image as a **matrix of pixel intensities** (grayscale).  
2. Compute the **SVD**: $X = U \Sigma V^T$.  
3. Reconstruct the image using only the **top $k$ singular values**, which keeps the most important structures while discarding less significant details.  
4. By varying $k$, we can **see how the image quality improves** as more singular values are included.  

This is a real-world example of **low-rank approximation** and shows why SVD is useful in image compression and dimensionality reduction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color

# ---- Load grayscale image ----
# Replace with any image path or URL
img = io.imread("https://www.snexplores.org/wp-content/uploads/sites/3/2019/11/860_animalmath_main.png")
img_gray = color.rgb2gray(img)
#red = img[:, :, 0]
#green = img[:, :, 1]
#blue = img[:, :, 2]

# Optional: downsample for faster computation
img_gray = img_gray[::2, ::2]

plt.figure(figsize=(6,6))
plt.imshow(img_gray, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis('off')
plt.show()

# ---- Compute SVD ----
U, S, Vt = np.linalg.svd(img_gray, full_matrices=False)

# ---- Function for rank-k reconstruction ----
def reconstruct_image(U, S, Vt, k):
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

# ---- Reconstruct images with different ranks ----
ranks = [5, 20, 50, 100]
plt.figure(figsize=(24,6))
for i, k in enumerate(ranks):
    Xk = reconstruct_image(U, S, Vt, k)
    plt.subplot(1, len(ranks), i+1)
    plt.imshow(Xk, cmap='gray')
    plt.title(f"k={k}")
    plt.axis('off')
plt.suptitle("Image Approximation with Rank-k SVD")
plt.show()

# ---- Plot singular values ----
plt.figure()
plt.semilogy(S, marker='o')
plt.xlabel("Index")
plt.ylabel("Singular Value")
plt.title("Singular Values of the Image")
plt.grid(True)
plt.show()

# Exercise(s): SVD





## Exercise 1: Homework check

### Tasks:
1. Verify your SVD findings for problems 17 and 18 of the homework.
2. If you want, try to visualize the geometry of both.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 2: Energy Capture with Singular Values

Singular values tell us how much “energy” or information each component of a matrix carries.  

In this exercise:

1. Compute the singular values of the above 10×10 matrix.  
2. Calculate the **cumulative energy** captured by the top k singular values:  

$$
\text{Energy captured by rank-}k = \frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{n} \sigma_i^2}
$$  

3. Plot the cumulative energy as a function of k.  
4. Determine the smallest k that captures at least 90% of the total energy. Compare this matrix to the original.
5. Apply this idea to the cat picture compresison above and display the 10%, 30% and 80% cumulative energy.

This exercise helps you **understand which singular values are most important** and how low-rank approximations retain the most significant structure in a matrix.

### Explain your findings here:




In [ ]:
# Write your code for the exercise here!

## Exercise 3: Directional Importance in Images (“Basis Images”)

In this exercise, you will explore **how each singular vector contributes to the overall image**.

### Tasks / Steps:

1. **Load a grayscale image** of your choice (or use the example image from the tutorial).  

2. **Compute the SVD** of the image matrix $X$:
   $$
   X = U \Sigma V^T
   $$

3. **Extract the top 3 right singular vectors** (columns of $V^T$, or rows of $V$).

4. **Form the rank-1 “basis images”** corresponding to these singular vectors:  
   $$
   \text{Basis image } i = \sigma_i \, u_i v_i^T
   $$  
   where $\sigma_i$ is the singular value and $u_i, v_i$ are the left and right singular vectors.  

5. **Reshape or scale the basis images** to match the original image dimensions for visualization.

6. **Plot the basis images** side by side.  
   - Observe what each basis image represents: large-scale structure, edges, patterns, or fine details.  
   - Note how the **magnitude of the singular value** indicates the importance of that component.  

### Guidance / Hints:

- Think of **basis images as building blocks**: combining the top few gives a rough approximation of the original image.  
- The **first basis image** usually captures the main structure or contrast.  
- The **second and third basis images** add details, orientation, or texture.  
- You can reconstruct a low-rank approximation of the image by summing the top k basis images:
  $$
  X_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T
  $$  
  Compare the reconstruction to the original to see how much information each component carries.  

### Explain your findings here:




In [ ]:
# Write your code for the exercise here!

## Exercise 4: Image Compression by Color Channel

In this exercise, you will explore how SVD can be applied **separately to each color channel** of an image.

### Tasks / Steps:

1. **Load a color image** of your choice (or use the example image from the tutorial).

2. **Separate the image into its Red, Green, and Blue channels**:
   - Red channel → `img[:, :, 0]`
   - Green channel → `img[:, :, 1]`
   - Blue channel → `img[:, :, 2]`

3. **Perform SVD on each channel independently**:
   $$
   X_c = U_c \Sigma_c V_c^T, \quad c \in \{R, G, B\}
   $$

4. **Reconstruct each channel using the top k singular values** (try multiple values of k, e.g., 5, 20, 50).

5. **Combine the reconstructed channels** back into a single color image and compare it with the original image.

6. **Plot the singular values for each channel** to visualize how the energy is distributed.

### Guidance / Hints:

- Use `np.linalg.svd` for each channel separately.  
- When reconstructing, remember to multiply `U @ np.diag(S[:k]) @ Vt` for each channel.  
- Use `np.stack([R_recon, G_recon, B_recon], axis=2)` to merge the channels back.  
- Observe: some channels may compress better than others depending on the image content.  


### Explain your findings here:




In [ ]:
# Write your code for the exercise here!

In [ ]:
# code to export notebook as .html for Canvas upload

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

notebook_name = "Lab_4"
!cp "/content/drive/MyDrive/Colab Notebooks/DSC 430/{notebook_name}.ipynb" /content/
!jupyter nbconvert --to html "/content/{notebook_name}.ipynb"
files.download(f"/content/{notebook_name}.html")

